In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')



In [7]:
df = pd.read_csv('rows.csv')

In [8]:
df.head()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
0,05/10/2019,Checking or savings account,Checking account,Managing an account,Problem using a debit or ATM card,NaN,NaN,NAVY FEDERAL CREDIT UNION,FL,328XX,Older American,NaN,Web,05/10/2019,In progress,Yes,NaN,3238275.0
1,05/10/2019,Checking or savings account,Other banking product or service,Managing an account,Deposits and withdrawals,NaN,NaN,BOEING EMPLOYEES CREDIT UNION,WA,98204,NaN,NaN,Referral,05/10/2019,Closed with explanation,Yes,NaN,3238228.0
2,05/10/2019,Debt collection,Payday loan debt,Communication tactics,Frequent or repeated calls,NaN,NaN,CURO Intermediate Holdings,TX,751XX,NaN,NaN,Web,05/10/2019,Closed with explanation,Yes,NaN,3237964.0
3,05/10/2019,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Old information reappears or never goes away,NaN,NaN,Ad Astra Recovery Services Inc,LA,708XX,NaN,NaN,Web,05/10/2019,Closed with explanation,Yes,NaN,3238479.0
4,05/10/2019,Checking or savings account,Checking account,Managing an account,Banking errors,NaN,NaN,ALLY FINANCIAL INC.,AZ,85205,NaN,NaN,Postal mail,05/10/2019,In progress,Yes,NaN,3238460.0


In [9]:
df.columns

Index(['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
       'Consumer complaint narrative', 'Company public response', 'Company',
       'State', 'ZIP code', 'Tags', 'Consumer consent provided?',
       'Submitted via', 'Date sent to company', 'Company response to consumer',
       'Timely response?', 'Consumer disputed?', 'Complaint ID'],
      dtype='object')

In [10]:

df = pd.read_csv(
    "rows.csv",
    usecols=["Product", "Consumer complaint narrative"],
    engine="python",
    on_bad_lines="skip"
)

print(df.shape)
df = df[df["Consumer complaint narrative"].notnull()]
print(df.shape)
print(df["Product"].value_counts())

(448885, 2)
(187186, 2)
Product
Credit reporting, credit repair services, or other personal consumer reports    79953
Debt collection                                                                 39355
Credit card or prepaid card                                                     18236
Mortgage                                                                        16528
Checking or savings account                                                     10863
Student loan                                                                     8648
Vehicle loan or lease                                                            4979
Money transfer, virtual currency, or money service                               4832
Payday loan, title loan, or personal loan                                        3792
Name: count, dtype: int64


In [13]:
samples_per_category = 2000

df_20k = (
    df.groupby("Product", group_keys=False)
      .apply(lambda x: x.sample(n=samples_per_category, random_state=42))
      .reset_index(drop=True)
)

print(df_20k.shape)
print(df_20k["Product"].value_counts())

(18000, 2)
Product
Checking or savings account                                                     2000
Credit card or prepaid card                                                     2000
Credit reporting, credit repair services, or other personal consumer reports    2000
Debt collection                                                                 2000
Money transfer, virtual currency, or money service                              2000
Mortgage                                                                        2000
Payday loan, title loan, or personal loan                                       2000
Student loan                                                                    2000
Vehicle loan or lease                                                           2000
Name: count, dtype: int64


In [14]:
df_20k.to_csv("complaints_20k.csv", index=False)

print("Saved successfully!")

Saved successfully!


In [16]:
df_20k.head()

,Product,Consumer complaint narrative
0,Checking or savings account,On XX/XX/2019 I went into my bank Wells Fargo ...
1,Checking or savings account,An inactive that I have had for many years was...
2,Checking or savings account,"opened a dispute for items i never received, c..."
3,Checking or savings account,The bank in question is XXXX XXXX XXXX. I requ...
4,Checking or savings account,Around XX/XX/XXXX! My account has been closed!...


In [17]:
df_20k.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18000 entries, 0 to 17999
Data columns (total 2 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   Product                       18000 non-null  object
 1   Consumer complaint narrative  18000 non-null  object
dtypes: object(2)
memory usage: 281.4+ KB


In [18]:
df_20k = df_20k.rename(columns={
    "Consumer complaint narrative": "text",
    "Product": "category"
})

df_20k.head()

,category,text
0,Checking or savings account,On XX/XX/2019 I went into my bank Wells Fargo ...
1,Checking or savings account,An inactive that I have had for many years was...
2,Checking or savings account,"opened a dispute for items i never received, c..."
3,Checking or savings account,The bank in question is XXXX XXXX XXXX. I requ...
4,Checking or savings account,Around XX/XX/XXXX! My account has been closed!...


In [19]:
print("Missing values:")
print(df_20k.isnull().sum())

print("\nDuplicate rows:")
print(df_20k.duplicated().sum())

Missing values:
category    0
text        0
dtype: int64

Duplicate rows:
53


In [20]:
df_20k = df_20k.drop_duplicates().reset_index(drop=True)

print(df_20k.shape)

(17947, 2)


In [21]:
df_20k['text'] = df_20k['text'].str.lower()

In [22]:
df_20k['text'] = df_20k['text'].str.replace(
    r'https?://\S+|www\.\S+', '', regex=True
)

In [23]:
df_20k['text'] = df_20k['text'].str.replace(
    r'<.*?>', '', regex=True
)

In [24]:
df_20k['text'] = df_20k['text'].str.replace(
    r'@\w+', '', regex=True
)

In [25]:
df_20k['text'] = df_20k['text'].str.replace(
    r'[^\w\s]', '', regex=True
)

In [26]:
df_20k['text'] = df_20k['text'].str.replace(
    r'[^\w\s]', '', regex=True
)

In [27]:
df_20k['text'] = df_20k['text'].str.replace(
    r'\d+', '', regex=True
)

In [29]:
df_20k['text'] = df_20k['text'].str.replace(
    r'\s+', ' ', regex=True
).str.strip()

In [30]:
df_20k.head()

,category,text
0,Checking or savings account,on xxxx i went into my bank wells fargo in xxx...
1,Checking or savings account,an inactive that i have had for many years was...
2,Checking or savings account,opened a dispute for items i never received ca...
3,Checking or savings account,the bank in question is xxxx xxxx xxxx i reque...
4,Checking or savings account,around xxxxxxxx my account has been closed bec...


In [31]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [32]:
stop_words = set(stopwords.words('english'))

In [33]:
def remove_stopwords(text):
    words = word_tokenize(text)
    words = [word for word in words if word not in stop_words]
    return ' '.join(words)

df_20k['text'] = df_20k['text'].apply(remove_stopwords)

In [34]:
df_20k[['category', 'text']].head()

,category,text
0,Checking or savings account,xxxx went bank wells fargo xxxx xxxx xxxx xxxx...
1,Checking or savings account,inactive many years cleaned identity theft man...
2,Checking or savings account,opened dispute items never received called td ...
3,Checking or savings account,bank question xxxx xxxx xxxx requested transfe...
4,Checking or savings account,around xxxxxxxx account closed girlfriend writ...


In [38]:
X = df_20k['text']
y = df_20k['category']


In [39]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [40]:
print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (14357,)
Testing: (3590,)


In [68]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf2 = TfidfVectorizer(
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_tfidf2 = tfidf2.fit_transform(X_train)
X_test_tfidf2 = tfidf2.transform(X_test)

print("Training:", X_train_tfidf2.shape)
print("Testing:", X_test_tfidf2.shape)



Training: (14357, 13382)
Testing: (3590, 13382)


In [65]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

y_encoded = encoder.fit_transform(y)

print("Classes:", encoder.classes_)
print("Encoded values:", y_encoded[:10])

Classes: ['Checking or savings account' 'Credit card or prepaid card'
 'Credit reporting, credit repair services, or other personal consumer reports'
 'Debt collection' 'Money transfer, virtual currency, or money service'
 'Mortgage' 'Payday loan, title loan, or personal loan' 'Student loan'
 'Vehicle loan or lease']
Encoded values: [0 0 0 0 0 0 0 0 0 0]


In [72]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
lr_model2 = LogisticRegression(max_iter=1000)

lr_model2.fit(X_train_tfidf2, y_train)

y_pred2 = lr_model2.predict(X_test_tfidf2)

print("Accuracy:", accuracy_score(y_test, y_pred2))
print("Precision:", precision_score(y_test, y_pred2, average='weighted'))
print("Recall:", recall_score(y_test, y_pred2, average='weighted'))
print("F1-Score:", f1_score(y_test, y_pred2, average='weighted'))

Accuracy: 0.8030640668523676
Precision: 0.804435775526716
Recall: 0.8030640668523676
F1-Score: 0.8032477707320292


In [73]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=2000)

param_grid = {
    'C': [0.01, 0.1, 1, 10, 100]
}

grid = GridSearchCV(
    lr,
    param_grid,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1
)

grid.fit(X_train_tfidf2, y_train)

print("Best Parameters:", grid.best_params_)
print("Best CV F1:", grid.best_score_)

Best Parameters: {'C': 1}
Best CV F1: 0.8085927779633435


In [74]:
best_lr = grid.best_estimator_

y_pred = best_lr.predict(X_test_tfidf2)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average='weighted'))
print("Recall:", recall_score(y_test, y_pred, average='weighted'))
print("F1-Score:", f1_score(y_test, y_pred, average='weighted'))

Accuracy: 0.8030640668523676
Precision: 0.804435775526716
Recall: 0.8030640668523676
F1-Score: 0.8032477707320292


In [76]:
import joblib

joblib.dump(best_lr, "complaint_model.pkl")
joblib.dump(tfidf2, "tfidf_vectorizer.pkl")
joblib.dump(encoder, "label_encoder.pkl")

print("Files saved successfully!")

Files saved successfully!
